# Repo Filter

Filter enriched repos from `selected_repos.jsonl` using parallel coordinates
(with test file counts) and browse the full dataset in an AG Grid table.

### Prerequisites

```bash
# Fetch + enrich
export GITHUB_TOKEN=ghp_...
uv run python evals/eda/fetch_repos.py
uv run python evals/eda/enrich_test_counts.py

# Notebook deps
pip install pandas numpy plotly ipywidgets ipyaggrid
```


In [ ]:
import json as _json
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display


In [ ]:
# Load enriched data from selected_repos.jsonl
jsonl_path = Path("selected_repos.jsonl")
if not jsonl_path.exists():
    jsonl_path = Path("evals/eda/selected_repos.jsonl")
if not jsonl_path.exists():
    raise FileNotFoundError(
        "Run fetch_repos.py and enrich_test_counts.py first"
    )

records = [_json.loads(line) for line in jsonl_path.read_text().splitlines() if line.strip()]
df = pd.DataFrame(records)

# Fill missing test count columns
for col in ["test_files_by_name", "files_in_test_dirs", "test_files_total"]:
    if col not in df.columns:
        df[col] = 0

print(f"Loaded {len(df)} repos")
df.head()


In [ ]:
# Language codes + log-scale columns
languages = sorted(df["language"].dropna().unique())
lang_map = {lang: i for i, lang in enumerate(languages)}
df["lang_code"] = df["language"].map(lang_map).fillna(-1).astype(int)

log_cols = [
    "stars",
    "size_mb",
    "recent_commits_90d",
    "test_files_total",
    "test_files_by_name",
    "files_in_test_dirs",
]
for col in log_cols:
    df[f"log_{col}"] = np.log1p(df[col])


In [ ]:
# ── Parallel coordinates plot ──

axes = [
    {
        "col": "lang_code",
        "label": "Language",
        "tickvals": list(lang_map.values()),
        "ticktext": list(lang_map.keys()),
    },
    {"col": "log_stars", "label": "Stars (log)"},
    {"col": "log_size_mb", "label": "Size MB (log)"},
    {"col": "log_recent_commits_90d", "label": "Recent Commits 90d (log)"},
    {"col": "log_test_files_total", "label": "Test Files (log)"},
    {"col": "log_test_files_by_name", "label": "Test Files by Name (log)"},
    {"col": "log_files_in_test_dirs", "label": "Files in Test Dirs (log)"},
]

dimensions = []
for ax in axes:
    d = {"label": ax["label"], "values": df[ax["col"]]}
    if "tickvals" in ax:
        d["tickvals"] = ax["tickvals"]
        d["ticktext"] = ax["ticktext"]
    dimensions.append(d)

fig = go.FigureWidget(
    data=go.Parcoords(
        line={
            "color": df["lang_code"],
            "colorscale": "Turbo",
            "showscale": True,
            "cmin": df["lang_code"].min(),
            "cmax": df["lang_code"].max(),
        },
        dimensions=dimensions,
        unselected={"line": {"color": "lightgray", "opacity": 0.3}},
    ),
    layout={
        "title": "Repo Filter - Parallel Coordinates",
        "width": 1200,
        "height": 600,
        "margin": {"l": 80, "r": 80, "t": 60, "b": 30},
    },
)

selection_output = widgets.Output(
    layout=widgets.Layout(max_height="400px", overflow_y="auto")
)
export_btn = widgets.Button(
    description="Export selected -> JSONL", button_style="success", icon="download"
)
status_label = widgets.Label(value="Drag ranges on axes above to filter repos")

TABLE_COLS = [
    "repo",
    "language",
    "stars",
    "size_mb",
    "recent_commits_90d",
    "test_files_total",
    "test_files_by_name",
    "files_in_test_dirs",
    "notes",
]


def _get_selected_mask() -> pd.Series:
    mask = pd.Series(True, index=df.index)
    parcoords = fig.data[0]
    for i, dim in enumerate(parcoords.dimensions):
        cr = dim.constraintrange
        if cr is not None:
            col = axes[i]["col"]
            if isinstance(cr[0], list | tuple):
                sub = pd.Series(False, index=df.index)
                for lo, hi in cr:
                    sub |= (df[col] >= lo) & (df[col] <= hi)
                mask &= sub
            else:
                lo, hi = cr
                mask &= (df[col] >= lo) & (df[col] <= hi)
    return mask


def _make_clickable(url: str) -> str:
    """Wrap a GitHub URL in an HTML anchor tag."""
    name = url.replace("https://github.com/", "")
    return f'<a href="{url}" target="_blank">{name}</a>'


def _update_selection(*_args):
    mask = _get_selected_mask()
    selected = df.loc[mask, TABLE_COLS].copy()
    with selection_output:
        clear_output(wait=True)
        status_label.value = f"{len(selected)} repos selected"
        if len(selected) > 0:
            show = selected.sort_values(["language", "stars"], ascending=[True, False]).reset_index(drop=True)
            show["repo"] = show["repo"].apply(_make_clickable)
            display(widgets.HTML(show.to_html(escape=False, index=False)))


for dim in fig.data[0].dimensions:
    dim.on_change(_update_selection, "constraintrange")


def _export_clicked(_btn):
    mask = _get_selected_mask()
    selected = df.loc[mask].copy()
    out_path = Path("filtered_repos.jsonl")
    with out_path.open("w") as f:
        for rank, (_, row) in enumerate(
            selected.sort_values(["language", "stars"], ascending=[True, False]).iterrows(), 1
        ):
            rec = {
                "rank": rank,
                "repo": row["repo"],
                "language": row.get("language", ""),
                "stars": int(row["stars"]),
                "size_mb": float(row["size_mb"]),
                "recent_commits_90d": int(row.get("recent_commits_90d", 0)),
                "test_files_total": int(row.get("test_files_total", 0)),
                "test_files_by_name": int(row.get("test_files_by_name", 0)),
                "files_in_test_dirs": int(row.get("files_in_test_dirs", 0)),
                "notes": row.get("notes", ""),
            }
            f.write(_json.dumps(rec) + "\n")
    status_label.value = f"Exported {len(selected)} repos -> {out_path}"


export_btn.on_click(_export_clicked)

display(
    widgets.VBox(
        [
            fig,
            widgets.HBox([status_label, export_btn]),
            selection_output,
        ]
    )
)

_update_selection()


## Full Dataset (AG Grid)

Sortable, filterable, resizable table with all columns. Falls back to
plain HTML if `ipyaggrid` is not installed.


In [ ]:
# ── Full data table (AG Grid if available, otherwise plain HTML) ──

GRID_COLS = [
    "rank",
    "repo",
    "language",
    "stars",
    "size_mb",
    "recent_commits_90d",
    "test_files_total",
    "test_files_by_name",
    "files_in_test_dirs",
    "notes",
]
_grid_df = (
    df[[c for c in GRID_COLS if c in df.columns]]
    .sort_values(["language", "stars"], ascending=[True, False])
    .reset_index(drop=True)
)

try:
    from ipyaggrid import Grid

    column_defs = []
    for col in _grid_df.columns:
        cd = {
            "headerName": col,
            "field": col,
            "sortable": True,
            "filter": True,
            "resizable": True,
        }
        if col == "repo":
            cd["width"] = 350
        elif col == "notes":
            cd["width"] = 400
        elif col == "language":
            cd["width"] = 120
        else:
            cd["width"] = 130
        column_defs.append(cd)

    grid = Grid(
        grid_data=_grid_df,
        grid_options={
            "columnDefs": column_defs,
            "enableSorting": True,
            "enableFilter": True,
            "enableColResize": True,
            "domLayout": "normal",
        },
        quick_filter=True,
        theme="ag-theme-balham",
        columns_fit="auto",
    )
    display(grid)

except ImportError:
    print("ipyaggrid not installed, falling back to full HTML table.")
    print("Install with: pip install ipyaggrid")
    _html_df = _grid_df.copy()
    _html_df["repo"] = _html_df["repo"].apply(
        lambda u: f'<a href="{u}" target="_blank">{u.replace("https://github.com/", "")}</a>'
    )
    display(widgets.HTML(_html_df.to_html(escape=False, index=False)))
